# Lesson 10 Lab — Roofline Reasoning before Tuning

**Puzzle:** When FLOPs, HBM bytes, arithmetic intensity, and ceilings change together, which observation tells you whether the kernel, layout, toolchain, or hardware boundary is responsible?

This notebook retains one complete RTX 5090 execution.


## Why this matters

This lab isolates FLOPs, HBM bytes, arithmetic intensity, and ceilings and keeps its comparison path explicit.


## 0. Predict before running

Predict correctness, warm latency ordering, and the first boundary case. Write what would disprove each prediction.


## 1. Theory and mechanism

Arithmetic intensity divides useful operations by required external bytes. Low-intensity pointwise kernels usually need fewer trips to memory; high-intensity matrix products need efficient tensor-core tiling. The model selects the next question but does not identify the achieved hardware bottleneck.


## 2. Trace the mechanism

```mermaid
flowchart LR
  A["Frozen input + contract"] --> B["FLOPs, HBM bytes, arithmetic intensity, and ceilings"]
  B --> C["Triton candidate"]
  B --> D["CUDA / library control"]
  C --> E["correctness + samples"]
  D --> E
  E --> F["bounded decision"]
```


## 3. Inspect the comparison boundary

Baseline: named PyTorch CUDA/library or standard-grid path. Candidate: reviewed Triton kernel or explicit model described below.

Quoting achieved TFLOP/s for a memory-bound copy, or GB/s for a compute-bound GEMM, hides the relevant ceiling.


## 4. Inspect the execution environment

The next cell asserts CUDA and records GPU, target, PyTorch, CUDA runtime, Triton, Python, and seed.


In [1]:
from pathlib import Path
import json, sys

ROOT = Path.cwd().parents[2]
sys.path.insert(0, str(ROOT / "scripts"))
from chapter05_runtime import environment, run_lesson

LESSON_NO = 10
LESSON_TITLE = 'Roofline Reasoning before Tuning'
ENV = environment(LESSON_NO)
print(json.dumps(ENV, indent=2, ensure_ascii=False))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "triton": "3.7.1",
  "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
  "python": "3.12.3",
  "seed": 20260823
}


## 5. Freeze the experiment

**Experiment:** Compute explicit algorithmic intensity for an affine kernel and a square GEMM, then measure each with its relevant unit.

Inputs, output contract, timer, and target stay fixed across compared paths.


## 6. Inspect and execute the reviewed code

The next cell calls the shared reviewed kernel source, retains full samples in `metrics`, checks maximum error, and prints the bounded analysis.


In [2]:
metrics, analysis_en, analysis_zh = run_lesson(LESSON_NO)
print(json.dumps(metrics, indent=2, ensure_ascii=False))
print(analysis_en)


{
  "primary": 0.25,
  "secondary": 170.66666666666666,
  "max_abs_error": 4.76837158203125e-07,
  "passed": true,
  "details": {
    "vector_median_ms": 0.021183999255299568,
    "matmul_median_ms": 0.013824000023305416,
    "vector_gbps": 1583.9517173135162,
    "matmul_tflops": 19.418074041337796,
    "vector_samples_ms": [
      0.033824000507593155,
      0.024351999163627625,
      0.021183999255299568,
      0.021183999255299568,
      0.020800000056624413,
      0.023264000192284584,
      0.021344000473618507,
      0.021888000890612602,
      0.0208320003002882,
      0.020800000056624413,
      0.019648000597953796,
      0.02300800010561943,
      0.019999999552965164,
      0.019872000440955162,
      0.01926399953663349,
      0.01929599978029728,
      0.021215999498963356,
      0.021215999498963356,
      0.021344000473618507,
      0.020096000283956528
    ],
    "matmul_samples_ms": [
      0.020160000771284103,
      0.015615999698638916,
      0.014527999795973301,

## 7. Read the retained RTX 5090 result

**Environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; Triton 3.7.1; Python 3.12.3.

| Measured field | Checked-in value |
|---|---:|
| Affine FLOP/byte | 0.2500 |
| GEMM FLOP/byte | 170.6667 |
| Maximum absolute error | 4.768e-07 |
| Acceptance gate | true |


## 8. Explain without overclaiming

The affine kernel has 0.250 FLOP/byte of requested traffic, while the square GEMM model has 170.7; they require different performance questions.

A named Triton or PyTorch CUDA path executed on the recorded GPU. The result applies to the printed shape, dtype, implementation, and software stack; internal hardware causes require profiler evidence.


## 9. Write the canonical artifact

The next cell stores the environment, full metrics, bilingual analysis, evidence label, and bounded conclusion.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": LESSON_NO,
    "title": LESSON_TITLE,
    "environment": ENV,
    "evidence_label": 'native-backend',
    "metrics": metrics,
    "analysis_en": analysis_en,
    "analysis_zh": analysis_zh,
    "conclusion": 'Use Roofline to classify the optimization direction; use counters and controlled variants to establish causality.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 10,
  "title": "Roofline Reasoning before Tuning",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "triton": "3.7.1",
    "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
    "python": "3.12.3",
    "seed": 20260823
  },
  "evidence_label": "native-backend",
  "metrics": {
    "primary": 0.25,
    "secondary": 170.66666666666666,
    "max_abs_error": 4.76837158203125e-07,
    "passed": true,
    "details": {
      "vector_median_ms": 0.021183999255299568,
      "matmul_median_ms": 0.013824000023305416,
      "vector_gbps": 1583.9517173135162,
      "matmul_tflops": 19.418074041337796,
      "vector_samples_ms": [
        0.033824000507593155,
        0.024351999163627625,
        0.021183999255299568,
        0.021183999255299568,
        0.020800000056624413,
        0.023264000192284584,
        0.021344000473618507,
        0.021888000890612602,


## 10. Make the bounded decision

> Use Roofline to classify the optimization direction; use counters and controlled variants to establish causality.

**Failure analysis:** Quoting achieved TFLOP/s for a memory-bound copy, or GB/s for a compute-bound GEMM, hides the relevant ceiling.


## 11. Extend and review

Add an awkward shape and non-contiguous layout. Stop on correctness failure. See `README.md` for references and the full review checklist.
